# DROP TARGET DB
#### Duplicate a source DB

In [1]:
import pymongo

def duplicate_db(source, target, uri="mongodb://localhost:27017/"):
    mongo_client = pymongo.MongoClient(uri)
    source_db = mongo_client[source]
    target_db = mongo_client[target]

    # Clear the Database if it Exists
    mongo_client.drop_database(target)


    for coll_name in source_db.list_collection_names():
        source_coll = source_db[coll_name]
        target_coll = target_db[coll_name]


        docs = list(source_coll.find({}))
        if docs:
            target_coll.insert_many(docs)
        print(f"Copied {len(docs)} documents from {coll_name}")

    print(f"Database '{source}' successfully duplicated to '{target}'")

duplicate_db("ceur_full", "ceur_full_etl")

Copied 25334 documents from papers
Copied 1910 documents from volumes
Database 'ceur_full' successfully duplicated to 'ceur_full_etl'


## Connection

In [7]:
from bson import ObjectId

client = pymongo.MongoClient("mongodb://localhost:27017/")

db = client['ceur_full_etl']


volumes = db['volumes']
papers = db['papers']
related_papers = db['related_papers']
all_papers = db['all_papers']

authors = db['authors']
authors_grouped = db['authors_grouped']

keywords = db['keywords']
keywords_final = db['keywords_final']

abstracts = db['abstracts']

# Create Collections

Related Papers Collection + Associate each Related Paper with an ID and to the Parent Paper.


In [8]:
# Clear the Collection if it Exists
related_papers.delete_many({})


pipeline = [
    {"$unwind": "$paper_info.related_papers"},
    {"$match": {
        "$expr": {
            "$gte": [
                {"$strLenCP": "$paper_info.related_papers.title"},
                20
            ]
        }
    }},
    {"$group": {
        "_id": "$paper_info.related_papers.title",
        "count": {"$sum": 1},
        "paper_ids": {"$push": "$_id"},
        "authors_set": {"$addToSet": "$paper_info.related_papers.authors"},
        "texts": {"$addToSet": "$paper_info.related_papers.text"}
    }},
    {"$sort": {"count": -1}},
    {"$addFields": {
        "cleaned_authors_sets": {
            "$map": {
                "input": "$authors_set",
                "as": "authors",
                "in": {
                    "$filter": {
                        "input": "$$authors",
                        "as": "a",
                        "cond": {
                            "$and": [
                                {"$gt": [{"$strLenCP": {"$trim": {"input": "$$a"}}}, 2]},
                                {"$not": [{"$in": [{"$toLower": "$$a"}, ["et al.", "et al"]]}]},
                                {"$not": [{"$regexMatch": {"input": "$$a", "regex": "BERT", "options": "i"}}]}
                            ]
                        }
                    }
                }
            }
        }
    }},
    {"$addFields": {
        "best_authors": {
            "$reduce": {
                "input": "$cleaned_authors_sets",
                "initialValue": [],
                "in": {
                    "$cond": [
                        {"$gt": [{"$size": "$$this"}, {"$size": "$$value"}]},
                        "$$this",
                        "$$value"
                    ]
                }
            }
        }
    }},
    {"$project": {
        "count": 1,
        "paper_ids": 1,
        "best_authors": 1,
        "texts": 1,
        "cleaned_authors_sets": 1
    }},
    {
        '$out': 'related_papers'
    }
]

results = list(papers.aggregate(pipeline))


Merge the Related Paper and Papers Collection into an All Papers Collection.

Since the Paper has more Information, the Related Paper is Merged into the Paper Collection.

In [9]:
# Clear the Collection if it Exists
all_papers.delete_many({})
title_to_id = dict()

pipeline_papers = [
    {
        "$project": {
            "paper_id": "$_id",
            "from": "paper",
            "count": {"$literal": 1},
            "title": 1,
            "_id": {
                "$function": {
                    "body": "function() { return new ObjectId(); }",
                    "args": [],
                    "lang": "js"
                }
            }
        }
    },
    {
        "$out": "all_papers"
    }
]

pipeline_rp = [
    {
        "$project": {
            "title": 1,
            "paper_id": "$_id",
            "from": { "$literal": "related" },
            "count": { "$literal": 1 },
            "_id": {
                "$function": {
                    "body": "function() { return new ObjectId(); }",
                    "args": [],
                    "lang": "js"
                }
            }
        }
    },
    {
        "$merge": {
            "into": "all_papers",
            "on": "_id"
        }
    }
]

papers.aggregate(pipeline_papers)
related_papers.aggregate(pipeline_rp)

# for paper in papers.find():
#     title_to_id[paper['title']] = paper['_id']
#     paper['paper_id'] = paper['_id']
#     paper['_id'] = ObjectId()
#     paper['from'] = 'paper'
#     paper['count'] = 1
#     all_papers.insert_one(paper)
#
#
# merged = 0
# for rpaper in related_papers.find():
#     if rpaper['title'] in title_to_id:
#         merged += 1
#         all_papers.update_one(
#             {'_id': title_to_id[rpaper['title']]},
#             {'$inc': {'count': rpaper['count']}}
#         )
#     else:
#         rpaper['paper_id'] = rpaper['_id']
#         rpaper['_id'] = ObjectId()
#         rpaper['from'] = 'related'
#         all_papers.insert_one(rpaper)

Authors Collection from the Related Papers and Papers Collections.

In [10]:
# Clear the Collection if it Exists
authors.delete_many({})

pipeline_related = [
    {"$unwind": "$best_authors"},
    {"$project": {
        "name": "$best_authors",
        "from": {"$literal": "related"},
        "related_id": "$_id"
    }}
]

pipeline_papers = [
    {"$unwind": "$author"},
    {"$project": {
        "name": "$author",
        "from": {"$literal": "paper"},
        "paper_id": "$_id"
    }}
]

pipeline = pipeline_related + [
    {"$unionWith": {
        "coll": "papers",
        "pipeline": pipeline_papers
    }},
    {"$addFields": {"_id": {"$function": {
        "body": "function() { return new ObjectId(); }",
        "args": [],
        "lang": "js"
    }}}},
    {"$out": "authors"}
]

related_papers.aggregate(pipeline)


# Clear the Collection if it Exists
authors_grouped.delete_many({})

pipeline = [
    {
        "$addFields": {
            "name_trimmed": {"$trim": {"input": "$name"}}
        }
    },
    {
        "$match": {
            "name_trimmed": {
                "$ne": None,
                "$not": {"$regex": r"\d"}
            }
        }
    },
    {
        "$group": {
            "_id": "$name_trimmed",
            "ids": {"$push": "$_id"},
            "from_set": {"$addToSet": "$from"},
            "paper_ids": {"$addToSet": "$paper_id"},
            "related_ids": {"$addToSet": "$related_id"}
        }
    },
    {
        "$out": "authors_grouped"
    }
]

authors.aggregate(pipeline)

Keywords Collection from Papers Collection

In [11]:
pipeline = [
    {
        "$unwind": "$paper_info.keywords"
    },
    {
        "$project": {
            "name": "$paper_info.keywords",
            "paper_id": "$_id"
        }
    },
    {
        "$group": {
            "_id": "$name",
            "paper_ids": {"$addToSet": "$paper_id"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "name": "$_id",
            "paper_ids": 1
        }
    },
    {
        "$merge": {
            "into": "keywords_final",
            "whenMatched": "replace",
            "whenNotMatched": "insert"
        }
    }
]

papers.aggregate(pipeline)

Abstract Collection from Papers Collection.


In [12]:
# Clear the Collection if it Exists
abstracts.delete_many({})

pipeline_abstract = [
    {
        "$match": {
            "abstract": {
                "$ne": None,
                "$not": {"$in": [""]},
               # "$expr": {"$gte": [{"$strLenCP": "$abstract"}, 50]}
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "text": "$abstract",
            "paper_id": "$_id"
        }
    },
    {
        "$addFields": {
            "_id": {"$function": {
                "body": "function() { return new ObjectId(); }",
                "args": [],
                "lang": "js"
            }}
        }
    },
    {
        "$out": "abstracts"
    }
]

papers.aggregate(pipeline_abstract)

# Create Memgraph

### Connection

In [23]:
from gqlalchemy import Memgraph

host_memgraph = "127.0.0.1"
port_memgraph = 7685
memgraph = Memgraph(host=host_memgraph, port=port_memgraph)


# Clear the Database
query_delete = """
    MATCH (n)
    DETACH DELETE n
"""

#memgraph.execute(query_delete)
#print("Database Cleared.")

def execute_batch(collection, query, batch_size = 10_000):
    batch = []
    batch_counter = 0
    for item in collection:
        item['mongo_id'] = str(item.pop('_id'))

        batch.append(item)

        if len(batch) >= batch_size:
            batch_counter += 1
            memgraph.execute(query, {"batch": batch})
            print(f"Inserted Batch {batch_counter} ({batch_size} Nodes)")

            batch = []

    if batch:
        batch_counter += 1
        memgraph.execute(query, {"batch": batch})
        print(f"Inserted Final Batch {batch_counter} ({len(batch)} items)")

Database Cleared.


## Nodes

In [24]:
## ADD Papers

all_papers_select = all_papers.find({}, {
    '_id': "$paper_id",
    'title': 1,
    'from': 1,
    'count': 1,
    'text': 1
})

query_add_papers = """
    UNWIND $batch AS row
    CREATE (:Paper {
        id: row.mongo_id,
        name: row.title,
        source: row.from,
        count: row.count,
        text: row.text
    })
"""

execute_batch(all_papers_select, query_add_papers)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Batch 14 (10000 Nodes)
Inserted Batch 15 (10000 Nodes)
Inserted Batch 16 (10000 Nodes)
Inserted Batch 17 (10000 Nodes)
Inserted Batch 18 (10000 Nodes)
Inserted Batch 19 (10000 Nodes)
Inserted Batch 20 (10000 Nodes)
Inserted Batch 21 (10000 Nodes)
Inserted Batch 22 (10000 Nodes)
Inserted Batch 23 (10000 Nodes)
Inserted Batch 24 (10000 Nodes)
Inserted Batch 25 (10000 Nodes)
Inserted Final Batch 26 (4273 items)


In [25]:
## ADD Authors

authors_select = authors_grouped.find({}, {
    '_id': 1,
})

query_add_authors = """
    UNWIND $batch as row
    CREATE(:Author {
        name: row.mongo_id
    })
"""

execute_batch(authors_select, query_add_authors)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Batch 14 (10000 Nodes)
Inserted Batch 15 (10000 Nodes)
Inserted Batch 16 (10000 Nodes)
Inserted Batch 17 (10000 Nodes)
Inserted Batch 18 (10000 Nodes)
Inserted Batch 19 (10000 Nodes)
Inserted Batch 20 (10000 Nodes)
Inserted Batch 21 (10000 Nodes)
Inserted Batch 22 (10000 Nodes)
Inserted Batch 23 (10000 Nodes)
Inserted Batch 24 (10000 Nodes)
Inserted Batch 25 (10000 Nodes)
Inserted Batch 26 (10000 Nodes)
Inserted Batch 27 (10000 Nodes)
Inserted Batch 28 (10000 Nodes)
Inserted Batch 29 (10000 Nodes)
Inserted Batch 30 (10000 Nodes)
Inserted Batch 31 (10000 Nodes)
Inserted Batch 32

In [26]:
## ADD Volumes

volumes_select = volumes.find({}, {
    '_id': 1,
    'title': 1,
    'pubyear': 1
})

query_add_volumes = """
    UNWIND $batch as row
    CREATE (:Volume {
        id: row.mongo_id,
        name: row.title,
        year: row.pubyear
    })
"""

execute_batch(volumes_select, query_add_volumes)

Inserted Final Batch 1 (1910 items)


In [39]:
## ADD Keywords

keywords_select = keywords_final.find({}, {
    '_id': 1,
    'name': 1,

})

query_add_keywords = """
    UNWIND $batch as row
    CREATE (:Keyword {
        id: row.mongo_id,
        name: row.name
    })
"""

execute_batch(keywords_select, query_add_keywords)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Final Batch 8 (1502 items)


In [28]:
## ADD Abstracts

abstracts_select = abstracts.find({}, {
    '_id': 1,
    'text': 1,

})

query_add_abstracts = """
    UNWIND $batch as row
    CREATE (:Abstract {
        id: row.mongo_id,
        name: row.text
    })
"""

execute_batch(abstracts_select, query_add_abstracts)

Inserted Batch 1 (10000 Nodes)
Inserted Final Batch 2 (5788 items)


In [29]:
## ADD Pubyears

pubyears = volumes.distinct('pubyear')

for item in pubyears:
    if item is not None:
        memgraph.execute("CREATE (:PubYear {year: $year})", {"year": item})

In [30]:
## ADD Indexes

memgraph.execute("CREATE INDEX ON :Paper(id)")
memgraph.execute("CREATE INDEX ON :Paper(name)")
memgraph.execute("CREATE INDEX ON :Author(name)")
memgraph.execute("CREATE INDEX ON :Volume(id)")
memgraph.execute("CREATE INDEX ON :Keyword(id)")
memgraph.execute("CREATE INDEX ON :Abstract(id)")

## Relationships

In [31]:
## WROTE — Author → Paper

query_paper_author = """
    UNWIND $batch AS row
    MATCH (a:Author {name: row.mongo_id})
    MATCH (p:Paper {id: row.target_id})
    MERGE (a)-[:WROTE]->(p);
"""

pipeline = [
    {
        "$project": {
            "ids": 1,
            "paper_ids": 1,
            "related_ids": 1
        }
    },
    {
        "$unwind": {
            "path": "$paper_ids",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$unwind": {
            "path": "$related_ids",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$project": {
            "_id": 1,
            "paper_id": "$paper_ids",
            "related_id": "$related_ids"
        }
    },
    {
        "$project": {
            "_id": 1,
            "target_id": {
                "$toString": {"$ifNull": ["$paper_id", "$related_id"]}
            }
        }
    },
    {
        "$match": {
            "target_id": {"$ne": None}
        }
    }
]

execute_batch(authors_grouped.aggregate(pipeline), query_paper_author)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Batch 14 (10000 Nodes)
Inserted Batch 15 (10000 Nodes)
Inserted Batch 16 (10000 Nodes)
Inserted Batch 17 (10000 Nodes)
Inserted Batch 18 (10000 Nodes)
Inserted Batch 19 (10000 Nodes)
Inserted Batch 20 (10000 Nodes)
Inserted Batch 21 (10000 Nodes)
Inserted Batch 22 (10000 Nodes)
Inserted Batch 23 (10000 Nodes)
Inserted Batch 24 (10000 Nodes)
Inserted Batch 25 (10000 Nodes)
Inserted Batch 26 (10000 Nodes)
Inserted Batch 27 (10000 Nodes)
Inserted Batch 28 (10000 Nodes)
Inserted Batch 29 (10000 Nodes)
Inserted Batch 30 (10000 Nodes)
Inserted Batch 31 (10000 Nodes)
Inserted Batch 32

In [32]:
## IN_VOLUME — Paper → Volume

query_paper_volume = """
    UNWIND $batch AS row
    MATCH (v:Volume {id: row.volume_id})
    MATCH (p:Paper {id: row.paper_id})
    MERGE (p)-[:IN_VOLUME]->(v);
"""

paper_volume_pipeline = [
    {
        "$match": {
            "volume_id": {"$ne": None}
        }
    },
    {
        "$project": {
            "paper_id": {"$toString": "$_id"},
            "volume_id": {"$toString": "$volume_id"}
        }
    }
]

execute_batch(papers.aggregate(paper_volume_pipeline), query_paper_volume)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Final Batch 3 (5334 items)


In [40]:
## HAS_KEYWORD — Paper → Keyword

query_paper_keyword = """
    UNWIND $batch AS row
    MATCH (k:Keyword {id: row.keyword_id})
    MATCH (p:Paper {id: row.paper_id})
    MERGE (p)-[:HAS_KEYWORD]->(k);
"""

paper_keyword_pipeline = [
    {
        "$unwind": "$paper_ids"
    },
    {
        "$project": {
            "keyword_id": {"$toString": "$_id"},
            "paper_id": {"$toString": "$paper_ids"}
        }
    }
]

execute_batch(keywords_final.aggregate(paper_keyword_pipeline), query_paper_keyword)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Final Batch 14 (1175 items)


In [34]:
## HAS_ABSTRACT — Paper → Abstract

query_paper_abstract = """
    UNWIND $batch AS row
    MATCH (a:Abstract {id: row.abstract_id})
    MATCH (p:Paper {id: row.paper_id})
    MERGE (p)-[:HAS_ABSTRACT]->(a);
"""

paper_abstract_pipeline = [
    {
        "$project": {
            "abstract_id": {"$toString": "$_id"},
            "paper_id": {"$toString": "$paper_id"}
        }
    }
]

execute_batch(abstracts.aggregate(paper_abstract_pipeline), query_paper_abstract)

Inserted Batch 1 (10000 Nodes)
Inserted Final Batch 2 (5788 items)


In [35]:
## PUBLISHED_IN — VOLUME → PUBYEAR

query_volume_pubyear = """
    UNWIND $batch AS row
    MATCH (v:Volume {id: row.volume_id})
    MATCH (py:PubYear {year: row.pubyear})
    MERGE (v)-[:PUBLISHED_IN]->(py);
"""

volume_pubyear_pipeline = [
    {
        "$match": {
            "pubyear": {"$ne": None}
        }
    },
    {
        "$project": {
            "volume_id": {"$toString": "$_id"},
            "pubyear": "$pubyear"
        }
    }
]

execute_batch(volumes.aggregate(volume_pubyear_pipeline), query_volume_pubyear)

Inserted Final Batch 1 (1909 items)


In [36]:
## CITES — Paper → Paper (Related)

query_paper_cites = """
    UNWIND $batch AS row
    MATCH (p1:Paper {id: row.source_id})
    MATCH (p2:Paper {name: row.target_title})
    MERGE (p1)-[r:CITES]->(p2);
"""

paper_paper_pipeline = [
    {
        "$unwind": "$paper_info.related_papers"
    },
    {
        "$project": {
            "source_id": {"$toString": "$_id"},
            "target_title": "$paper_info.related_papers.title"
        }
    },
    {
        "$match": {
            "target_title": {
                "$ne": None,
                "$not": {"$in": ["", "N/A", "na", "n/a"]}
            }
        }
    }
]

execute_batch(papers.aggregate(paper_paper_pipeline), query_paper_cites)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Batch 14 (10000 Nodes)
Inserted Batch 15 (10000 Nodes)
Inserted Batch 16 (10000 Nodes)
Inserted Batch 17 (10000 Nodes)
Inserted Batch 18 (10000 Nodes)
Inserted Batch 19 (10000 Nodes)
Inserted Batch 20 (10000 Nodes)
Inserted Batch 21 (10000 Nodes)
Inserted Batch 22 (10000 Nodes)
Inserted Batch 23 (10000 Nodes)
Inserted Batch 24 (10000 Nodes)
Inserted Batch 25 (10000 Nodes)
Inserted Final Batch 26 (2412 items)
